# Bài 8: Lập trình hướng đối tượng (OOP)

OOP là cách **gom dữ liệu và hành vi liên quan vào một "đối tượng"**. Với AI Engineer, đây là bài cực kỳ quan trọng: mọi model trong PyTorch là một `class` kế thừa `nn.Module`, mọi mô hình scikit-learn là một object có `.fit()` và `.predict()`, `Dataset`, `DataLoader`, `Optimizer`... đều là class. Hiểu OOP là bạn đọc hiểu được code của các thư viện này.

Mục tiêu bài học:
- `class`, `__init__`, thuộc tính (attribute) và phương thức (method), `self`
- Phương thức đặc biệt (dunder): `__str__`, `__repr__`, `__len__`
- Thuộc tính lớp vs thuộc tính đối tượng
- Kế thừa (inheritance), `super()`, ghi đè (override)
- Đóng gói (encapsulation) & `@property`
- Ví dụ thực tế: một class "model" mini có `fit` / `predict` giống scikit-learn

> Nhắc lại cách học: đọc lý thuyết → chạy thử ví dụ → tự làm bài tập, đừng xem lời giải trước.

## 1. Class và object

- **Class**: bản thiết kế (khuôn).
- **Object (instance)**: một sản phẩm cụ thể tạo từ khuôn đó.

`__init__` là hàm **khởi tạo**, tự chạy khi tạo object. `self` là chính object đang được tạo/gọi — luôn là tham số đầu tiên của mọi method.

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name      # thuộc tính của đối tượng
        self.age = age

    def bark(self):           # phương thức
        print(f"{self.name} says: Gâu gâu!")

d1 = Dog("Milu", 3)
d2 = Dog("Bông", 5)

print(d1.name, d1.age)
d1.bark()
d2.bark()
print(type(d1))

## 2. Method thay đổi trạng thái

Điểm mạnh của OOP: object **nhớ trạng thái** giữa các lần gọi method. Ví dụ một "bộ theo dõi loss" khi huấn luyện:

In [ ]:
class LossTracker:
    def __init__(self):
        self.losses = []          # trạng thái ban đầu

    def add(self, loss):
        self.losses.append(loss)

    def average(self):
        return sum(self.losses) / len(self.losses)

    def best(self):
        return min(self.losses)

tracker = LossTracker()
for loss in [0.82, 0.65, 0.41, 0.30]:
    tracker.add(loss)

print("Trung bình:", tracker.average())
print("Tốt nhất:", tracker.best())
print(tracker.losses)

## 3. Phương thức đặc biệt (dunder methods)

Tên bao bởi hai dấu gạch dưới `__xxx__` được Python gọi tự động:

| Method | Được gọi khi |
|---|---|
| `__init__` | tạo object |
| `__str__` | `print(obj)` / `str(obj)` — hiển thị cho người dùng |
| `__repr__` | hiển thị trong notebook / debug |
| `__len__` | `len(obj)` |
| `__getitem__` | `obj[i]` — chính là thứ `Dataset` của PyTorch dùng! |

In [ ]:
class Dataset:
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

    def __repr__(self):
        return f"Dataset(n={len(self)})"

ds = Dataset(["img1", "img2", "img3"])
print(len(ds))      # gọi __len__
print(ds[1])        # gọi __getitem__
print(ds)           # gọi __repr__ (khi không có __str__)
for item in ds:     # có __getitem__ là duyệt được
    print(item)

> Đây đúng là hình dạng tối giản của `torch.utils.data.Dataset`: bạn chỉ cần viết `__len__` và `__getitem__`. Sang Module 6 bạn sẽ thấy lại.

## 4. Thuộc tính lớp vs thuộc tính đối tượng

- Thuộc tính **đối tượng** (`self.x`): mỗi object một giá trị riêng.
- Thuộc tính **lớp** (khai báo ngay trong class, ngoài method): **dùng chung** cho mọi object.

In [ ]:
class Model:
    framework = "pytorch"          # thuộc tính lớp: chung cho mọi Model

    def __init__(self, name):
        self.name = name           # thuộc tính đối tượng: riêng từng cái

m1 = Model("resnet")
m2 = Model("bert")
print(m1.name, m1.framework)
print(m2.name, m2.framework)
print(Model.framework)             # truy cập qua tên class cũng được

## 5. Kế thừa (inheritance)

Class con **thừa hưởng** thuộc tính/method của class cha, rồi thêm hoặc **ghi đè (override)** theo nhu cầu. `super()` gọi lại phiên bản của class cha. Cú pháp: `class Con(Cha):`.

Đây chính là cách bạn viết model PyTorch: `class MyNet(nn.Module):` rồi gọi `super().__init__()`.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

class Cat(Animal):
    def __init__(self, name, color):
        super().__init__(name)     # gọi __init__ của cha để gán self.name
        self.color = color

    def speak(self):               # ghi đè
        return "Meo!"

class Cow(Animal):
    def speak(self):
        return "Moo!"

animals = [Cat("Tom", "xám"), Cow("Bò Sữa"), Animal("Lạ")]
for a in animals:
    print(a.name, "->", a.speak())   # cùng lời gọi, hành vi tuỳ loại (đa hình)

print(isinstance(animals[0], Animal))   # Cat là một Animal -> True

## 6. Đóng gói & `@property`

Quy ước Python: tên bắt đầu bằng `_` (ví dụ `self._lr`) nghĩa là "nội bộ, đừng đụng trực tiếp". `@property` cho phép truy cập như thuộc tính nhưng vẫn **kiểm tra/tính toán** phía sau.

In [ ]:
class Optimizer:
    def __init__(self, lr):
        self.lr = lr               # đi qua setter bên dưới

    @property
    def lr(self):
        return self._lr

    @lr.setter
    def lr(self, value):
        if value <= 0:
            raise ValueError("learning rate phải > 0")
        self._lr = value

opt = Optimizer(0.01)
print(opt.lr)
opt.lr = 0.001      # dùng như thuộc tính bình thường
print(opt.lr)

try:
    opt.lr = -1
except ValueError as e:
    print("Lỗi:", e)

## 7. Ví dụ tổng hợp: model mini kiểu scikit-learn

Mô hình scikit-learn nào cũng có `.fit(X, y)` để học và `.predict(X)` để dự đoán. Dưới đây là model đơn giản nhất: **luôn dự đoán nhãn xuất hiện nhiều nhất** trong dữ liệu huấn luyện (baseline).

In [ ]:
class MajorityClassifier:
    def __init__(self):
        self.majority_label = None

    def fit(self, X, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        self.majority_label = max(counts, key=counts.get)
        return self                    # trả về self để gọi nối chuỗi như sklearn

    def predict(self, X):
        return [self.majority_label for _ in X]

    def score(self, X, y):
        preds = self.predict(X)
        return sum(p == t for p, t in zip(preds, y)) / len(y)

X_train = [[1], [2], [3], [4], [5]]
y_train = ["cat", "dog", "cat", "cat", "dog"]

model = MajorityClassifier().fit(X_train, y_train)
print(model.predict([[10], [20]]))
print("accuracy:", model.score(X_train, y_train))

## Bài tập thực hành

1. Tạo class `Rectangle` có `__init__(self, width, height)`, method `area()` (diện tích) và `perimeter()` (chu vi). Tạo hình `3 x 4` và in diện tích, chu vi.
2. Tạo class `Counter` có thuộc tính `count = 0` trong `__init__`, method `increment()` (tăng 1) và `reset()` (về 0). Gọi `increment()` 3 lần rồi in `count`; sau đó `reset()` và in lại.
3. Tạo class `Playlist` lưu list `songs`, cài `__len__` (số bài) và `__getitem__` (lấy bài theo chỉ số). Tạo playlist 3 bài, in `len(...)`, bài đầu tiên, và duyệt bằng `for`.
4. Tạo class `Student` (có `name`, `score`) và cài `__repr__` trả về chuỗi dạng `Student(name='An', score=8.5)`. Tạo 1 student rồi `print` nó.
5. Tạo class cha `Shape` có method `area()` trả `0`. Tạo class con `Circle(Shape)` (nhận `radius`, dùng `3.14159`) và `Square(Shape)` (nhận `side`), ghi đè `area()`. Với list `[Circle(1), Square(2)]`, in diện tích từng hình.
6. **(Nâng cao)** Viết class `MeanRegressor` giống mẫu ở mục 7: `fit(X, y)` lưu **trung bình của y** vào `self.mean_`, `predict(X)` trả list gồm `mean_` lặp lại cho mỗi phần tử của X, và `mse(X, y)` trả sai số bình phương trung bình giữa dự đoán và `y`. Thử với `y = [2, 4, 6, 8]` → `mean_ = 5.0`, `mse = 5.0`.

In [ ]:
# Bài 1: Rectangle

In [ ]:
# Bài 2: Counter

In [ ]:
# Bài 3: Playlist với __len__ và __getitem__

In [ ]:
# Bài 4: Student với __repr__

In [ ]:
# Bài 5: Shape, Circle, Square

In [ ]:
# Bài 6 (nâng cao): MeanRegressor

## Gợi ý

- Bài 1: nhớ viết `self.width`, `self.height` trong `area()` và `perimeter()`.
- Bài 3: `__len__` trả `len(self.songs)`, `__getitem__` trả `self.songs[index]`.
- Bài 4: `return f"Student(name={self.name!r}, score={self.score})"` — `!r` hiển thị chuỗi có dấu nháy.
- Bài 5: class con không cần viết `__init__` của cha nếu cha không có; chỉ cần `__init__` riêng của con.
- Bài 6: `mse = sum((p - t) ** 2 for p, t in zip(preds, y)) / len(y)`.

## Lời giải mẫu

⚠️ Chỉ chạy ô dưới đây **sau khi** bạn đã tự làm xong 6 bài tập ở trên.

In [ ]:
# --- Lời giải mẫu ---

# Bài 1
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(3, 4)
print("Diện tích:", r.area(), "| Chu vi:", r.perimeter())

# Bài 2
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1

    def reset(self):
        self.count = 0

c = Counter()
for _ in range(3):
    c.increment()
print(c.count)
c.reset()
print(c.count)

# Bài 3
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, index):
        return self.songs[index]

p = Playlist(["A", "B", "C"])
print(len(p), p[0])
for song in p:
    print(song)

# Bài 4
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    def __repr__(self):
        return f"Student(name={self.name!r}, score={self.score})"

print(Student("An", 8.5))

# Bài 5
class Shape:
    def area(self):
        return 0

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

for shape in [Circle(1), Square(2)]:
    print(type(shape).__name__, shape.area())

# Bài 6
class MeanRegressor:
    def __init__(self):
        self.mean_ = None

    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]

    def mse(self, X, y):
        preds = self.predict(X)
        return sum((p - t) ** 2 for p, t in zip(preds, y)) / len(y)

y = [2, 4, 6, 8]
X = [[1], [2], [3], [4]]
reg = MeanRegressor().fit(X, y)
print("mean_ =", reg.mean_, "| mse =", reg.mse(X, y))

## Tổng kết

Bạn đã học được:
- ✅ `class`, `__init__`, `self`, thuộc tính và phương thức; object nhớ trạng thái
- ✅ Dunder methods: `__repr__`, `__len__`, `__getitem__` (nền tảng của `Dataset` trong PyTorch)
- ✅ Thuộc tính lớp vs thuộc tính đối tượng
- ✅ Kế thừa, `super()`, ghi đè method, đa hình
- ✅ `@property` để kiểm tra giá trị khi gán
- ✅ Thiết kế class có `fit` / `predict` giống scikit-learn

**Bài tiếp theo:** Bài 9 — Xử lý lỗi (exception) & làm việc với file: `try/except`, đọc/ghi file text, `with open(...)`. Làm xong bài tập ở trên rồi báo mình nhé!